## Sklearn Usage & Comparision with fake data
#### Goal : Verify sklearn matches my scratch GD on identical fake data

In [24]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import mlflow

In [25]:
np.random.seed(42)
n = 100
X = np.random.uniform(0, 10, n)

y = 3 * X + 5 + np.random.normal(0, 2, n) #True_w = 3, True_b = 5
print(X[:5])
print(y[:5])

[3.74540119 9.50714306 7.31993942 5.98658484 1.5601864 ]
[16.4102977  32.92341449 27.14333981 18.9846167   9.24121544]


In [27]:
mlflow.set_experiment('sklearn_usage_part_a')
with mlflow.start_run(run_name='part_a_fakedata'):
    model = LinearRegression()
    X = X.reshape(-1, 1)
    model.fit(X=X,y=y)
    mlflow.log_param('n', n)
    mlflow.log_metric('coef', model.coef_[0])
    mlflow.log_metric('intercept', model.intercept_)
    mlflow.sklearn.log_model(model,name='model')
    #the scratch_w and scratch_b hardcoded values taken from 03-linreg-from-scratch-part2.ipynb ---Final w and b
    print(f'sk_learn w = {model.coef_[0]:.4f} | scratch_w = 2.911')
    print(f'sk_learn b = {model.intercept_:.4f} | scratch_b = 5.411')


2026/05/03 21:30:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/03 21:30:22 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


sk_learn w = 2.9080 | scratch_w = 2.911
sk_learn b = 5.4302 | scratch_b = 5.411


In [28]:
print(np.isclose([2.9], [2.9]))
print(np.isclose([5.43], [5.41]))

print(np.isclose(model.coef_[0], 2.911, atol=0.01))
print(np.isclose(model.intercept_, 5.411, atol=0.01))

[ True]
[False]
True
False


Whatever i have done in scratch pad in 30lines, slearn did it in 3lines --- That is awesome.

But one key point to be noted and learn is that what the sklearn is doing underthe hood needs to be learn.

"sklearn doesn't iterate at all — it solves the Normal Equation, a closed-form linear-algebra formula that gives the optimal w and b in one shot. Gradient descent (what I did from scratch) is the iterative alternative — necessary when the dataset is too big for the matrix inverse to be feasible."

In [29]:
from sklearn.datasets import fetch_california_housing
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import mlflow

In [30]:
housing = fetch_california_housing()
print(f'The housing data has following features : {housing.feature_names}')
print(f'\nThe housing data describe as below: \n{housing.DESCR}')
print(f'\nThe housing data shape : {housing.data.shape}')
print(f'\nThe shape of target variable in the housing data :  {housing.target.shape}')

The housing data has following features : ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']

The housing data describe as below: 
.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib repository.
https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

The target variable is t

In [31]:
print(f'The housing data has following features : {housing.feature_names}')
df = pd.DataFrame(housing.data, columns=housing.feature_names)
print('\n The description of the data after loading in dataframe.')
print(df.describe())
print('\nThe head of the data frame')
print(df.head())
print('\nThe tail of the data frame')
print(df.tail())


The housing data has following features : ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']

 The description of the data after loading in dataframe.
             MedInc      HouseAge      AveRooms     AveBedrms    Population  \
count  20640.000000  20640.000000  20640.000000  20640.000000  20640.000000   
mean       3.870671     28.639486      5.429000      1.096675   1425.476744   
std        1.899822     12.585558      2.474173      0.473911   1132.462122   
min        0.499900      1.000000      0.846154      0.333333      3.000000   
25%        2.563400     18.000000      4.440716      1.006079    787.000000   
50%        3.534800     29.000000      5.229129      1.048780   1166.000000   
75%        4.743250     37.000000      6.052381      1.099526   1725.000000   
max       15.000100     52.000000    141.909091     34.066667  35682.000000   

           AveOccup      Latitude     Longitude  
count  20640.000000  20640.000000  2064

What is the target variable? What are its units?
- The target variable will be the value of the house based on the feature selection. so i mean if you slect the feature like 'HouseAge', 'AveRooms', 'AveBedrms' then based on these feature selection what will be the predicted price and you can change these combinations and can predict what will be the price of house? And units will be - units of 100,000 (The unit can be $ or INR which ever you consider).

Which 2 features do you predict will matter most? Why? (Write your guess BEFORE fitting. This is the experiment.
- Predicting MedInc and AveOccup will be matter most because if iam considering which seeing the house value i will look into two things which i have mentioned.


In [8]:
X, y = fetch_california_housing(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(X, y,  test_size=0.2, random_state=42)

print(f'X_train shape : {X_train.shape}')
print(f'X_test shape : {X_test.shape}')
print(f'y_train shape : {y_train.shape}')
print(f'y_test shape : {y_test.shape}')

X_train shape : (16512, 8)
X_test shape : (4128, 8)
y_train shape : (16512,)
y_test shape : (4128,)


## Why split at all?
- Splitting data is very important because when you are training the model you will strictly tain based on training data to learn model patterns not the data itself. so, you are not showing test data to the model so you will verify with test data whether model has learned the patterns or it has simply learned the training data itself. If the model truley learns the pattern then model will perform well on the test data

In [34]:
mlflow.set_experiment('sklearn-california-mlflow')
with mlflow.start_run(run_name='california-housing-data'):
    #Model Training
    model = LinearRegression()
    model.fit(X=X_train, y=y_train)

    # No reshape needed because X_train already has 8 columns

    #Model evalution
    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    r2score = r2_score(y_test, y_pred)
    mlflow.log_param('X_train_shape', X_train.shape)
    mlflow.log_param('test_size', 0.2)
    mlflow.log_param('random_state', 42)

    mlflow.log_metric('mse', mse)
    mlflow.log_metric('r2_score', r2score)
    mlflow.log_metric('RMSE', np.sqrt(mse))
    mlflow.sklearn.log_model(model, name='model')

    print(f'MSE : {mse:.4f}')
    print(f'RMSE : {np.sqrt(mse):.4f}')
    print(f'r2 score : {r2score:.4f}')


2026/05/03 22:00:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/03 22:00:49 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


MSE : 0.5559
RMSE : 0.7456
r2 score : 0.5758


## MSE:
- The mean squared error is 0.559 which quite difficult to interpret because we have squared the errors. so, what we have to do is find the RMSE in which the units will be in same as y, so it will be easy to interpret.
So we have got the value RMSE : 0.7456 ---> If we multiply with the units of y i.e., 100000 * 0.7456 = 74560units which means that the predictions will be off by 74560units.

## R**2:
R2 value tells us how much we can trust the model predictions, the value we got 0.5758, so what it means is that 50% of variance of the features are explained w.r.t to house value by the model.

In [11]:
# The detective work: interpret coefficients
#['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
print(f'W MedInc: {model.coef_[0]:.4f}')
print(f'W HouseAge: {model.coef_[1]:.4f}')
print(f'W AveRooms: {model.coef_[2]:.4f}')
print(f'W AveBedrms: {model.coef_[3]:.4f}')
print(f'W Population: {model.coef_[4]:.4f}')
print(f'W AveOccup: {model.coef_[5]:.4f}')
print(f'W Latitude: {model.coef_[6]:.4f}')
print(f'W Longitude: {model.coef_[7]:.4f}')
print(f'b : {model.intercept_}')

W MedInc: 0.4487
W HouseAge: 0.0097
W AveRooms: -0.1233
W AveBedrms: 0.7831
W Population: -0.0000
W AveOccup: -0.0035
W Latitude: -0.4198
W Longitude: -0.4337
b : -37.02327770606397


# The detective work: interpret coefficients

Which feature has the largest positive coefficient? Does that make intuitive sense?
- feature : AveBedrms has the hight co-efficient in raw magnitude, but you can not compare the coefficients which are in different scales, inorder to compare the coefficients first you need to standardize them and then you can compare. MedInc is likely to be the real driver because it varies over a much larger range.


Which feature has a negative coefficient? Why might that be? (Especially: Latitude and Longitude will look weird — think geographically about California. North vs south. Coast vs inland.)
- There are multiple features has negative coefficients, AveRooms, Population, AveOccup and Latitude and Longitude,
To understand the negative coefficients will be looking for more examples in week-2 as more exploration will be done.


Any coefficient that surprises you? (AveRooms or AveBedrms often do. Think about why — hint: they're correlated with each other and with income. Multicollinearity is a Week 2 topic, but you can name-drop the curiosity here.)
- AverRooms and AveBedRms are suprising for me because if the AveBedrms are increasing then then the AveRooms also should increase because im considering the assumption that bedrooms and room are directly proporstional.


Compare to your prediction from Step 1 — was your guess right?
- No, my prediction is wrong what i have predicted in frist step is the MedInc and AveOccup, MedInc is second highest, but my prediction is off for AveOccup.

`NOTE:` The doubt and what i'm curious about how do i interpret the negative coefficients, what does this mean in real world considerations. And i wanted Cluade to redraw the detective work. so that i can compare these things with my answers.

## TakeAway Narrative
How did this differ from the fake-data experience?
- There is a lot of difference i can see while working with fake data and real world data, one of those things is the when working with fake data you know what will be the coefficient value and there is only feature in fake data i have worked, but in real world data there are multiple features and the way predictions also differ interms what are the units will be for y.

What did the coefficients teach you about California housing?
- The coefficients in california housing teaches that the coefficient can be negative and we have to deal with it, and the coefficients can be colinear in which the coefficients will have dependent relationship between them

What's the honest verdict on this model? 
- The R**2 ~ 0.6 which means the only 60Percent variance of house value w.r.t features will be explained, but this is the base model in which the model can be improved later.


Why does MLflow matter?
- MLflow matters because in tradition software developement system logging wont work here because you will be running the model so many times and in traditional logging system it is difficult get what parameter do we set to get the r2 score 0.8 or something. mlflow provides all these data, what parameters do we set, what are the metrics we got and tracing data everything related debug data will be provided by MLflow.

What would have happened if you ran this without MLflow and came back in 2 weeks?
- You will be get lost on what params gave the highest r2score and its like developing the software without having git on the system 